# Practica 2 Aprendizaje de Pytorch
## Integrantes: Ana Martinez Albendea, Nicolás García Moncho, Rebeca Sánchez Denegri

## Introducción

El objetivo de la práctica es desarrollar un modelo de clasificación utilizando PyTorch y poder entrenar una red neuronal para que sea lo mas eficiente posible.

## Análisis del Dataset

El dataset utilizado está compuesto por información de pacientes que incluye tanto variables categóricas como numéricas. La variable objetivo es binaria, indicando la presencia o ausencia de diabetes. Para preparar los datos para el modelo, se realizó una codificación de las variables categóricas (GenHlth, Age, Education, Income) mediante One-Hot Encoding, lo que permitió convertirlas en representaciones numéricas. Adicionalmente, las variables numéricas (BMI, MentHlth, PhysHlth) fueron normalizadas para garantizar que todas las características estuvieran en la misma escala, mejorando la eficiencia y estabilidad del entrenamiento del modelo.

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch.nn as nn
from sklearn.preprocessing import OneHotEncoder

## Carga y exploración de datos

Comenzamos cargando el dataset y realizando una muestra aleatoria del 20% para trabajar con un subconjunto de datos con el fin de optimizar los tiempos de procesamiento.

El dataset contiene un total de 21 variables, donde algunas son categoricas, como *GenHlth* o *Age*, y otras numericas, como *MenHlth*, además de la variable objetivo *Diabetes_binary*.

In [2]:
df = pd.read_csv('archive/diabetes_reduced.csv')
df = df.sample(frac=0.2, random_state=42)
df.tail()

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
29547,0.0,0.0,0.0,1.0,32.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,3.0,1.0,0.0,0.0,1.0,4.0,4.0,8.0
59313,1.0,1.0,0.0,1.0,20.0,0.0,1.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,13.0,5.0,3.0
44805,1.0,1.0,1.0,1.0,27.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,2.0,0.0,13.0,0.0,0.0,11.0,5.0,5.0
30661,0.0,0.0,0.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,1.0,1.0,8.0,4.0,6.0
30295,0.0,1.0,1.0,1.0,19.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,13.0,4.0,5.0


## Procesamiento

## Codificación de variables Categóricas

Las variables categóricas fueron convertidas a representación One-Hot Encoding utilizando *OneHotEncoder*. Consiste en crear una nueva columna para cada categoría de la variable original. Si una instancia pertenece a una categoría, se asigna un valor de 1 en la columna correspondiente, y un valor de 0 en todas las demás columnas. Esto permite representar de manera binaria las categorías sin implicar ningún orden entre ellas.
En nuestro caso utilizamos la libreria sklearn.preprocessing que nos aporta un método para implementar este cambio.

In [3]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

encoder = OneHotEncoder(sparse_output=False)

encoded = encoder.fit_transform(df[['GenHlth','Age','Education','Income']])

encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(['GenHlth','Age','Education','Income']))
df_one_hot = pd.concat([df, encoded_df], axis=1).drop(columns=['GenHlth','Age','Education','Income'])



###  División en Conjuntos de Entrenamiento y Prueba

El dataset se divide en un 80% para entrenamiento y un 20% para prueba.

In [4]:
import sklearn
import sklearn.model_selection


df_train, df_test = sklearn.model_selection.train_test_split(df, train_size=0.8, random_state=1)
train_stats = df_train.describe().transpose()
train_stats

,count,mean,std,min,25%,50%,75%,max
Diabetes_binary,11310.0,0.496905,0.500013,0.0,0.0,0.0,1.0,1.0
HighBP,11310.0,0.560566,0.496340,0.0,0.0,1.0,1.0,1.0
HighChol,11310.0,0.526525,0.499318,0.0,0.0,1.0,1.0,1.0
CholCheck,11310.0,0.975508,0.154576,0.0,1.0,1.0,1.0,1.0
BMI,11310.0,29.738992,7.207996,13.0,25.0,28.0,33.0,98.0
Smoker,11310.0,0.474978,0.499396,0.0,0.0,0.0,1.0,1.0
Stroke,11310.0,0.062776,0.242571,0.0,0.0,0.0,0.0,1.0
HeartDiseaseorAttack,11310.0,0.151194,0.358253,0.0,0.0,0.0,0.0,1.0
PhysActivity,11310.0,0.696198,0.459918,0.0,0.0,1.0,1.0,1.0
Fruits,11310.0,0.602653,0.489371,0.0,0.0,1.0,1.0,1.0


### Normalización de Variables Numéricas

Las columnas numéricas es necesario normalizarlas para tener una media de 0 y una desviación estándar de 1. Esto asegura que las características numéricas esten en la misma escala, mejorando la estabilidad del modelo. 

In [5]:
from packaging import version


numeric_column_names = ['BMI', 'MentHlth', 'PhysHlth']
df_train_norm, df_test_norm = df_train.copy(), df_test.copy()
if version.parse(pd.__version__) >= version.parse("2.0.0"):

    for col_name in numeric_column_names:
        mean = train_stats.loc[col_name, 'mean']
        std = train_stats.loc[col_name, 'std']
        df_train_norm[col_name] = (df_train_norm[col_name] - mean) / std
        df_test_norm[col_name] = (df_test_norm[col_name] - mean) / std

else:

    for col_name in numeric_column_names:
        mean = train_stats.loc[col_name, 'mean']
        std  = train_stats.loc[col_name, 'std']
        df_train_norm.loc[:, col_name] = (df_train_norm.loc[:, col_name] - mean) / std
        df_test_norm.loc[:, col_name] = (df_test_norm.loc[:, col_name] - mean) / std
        
df_train_norm.tail()



,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
64625,1.0,1.0,1.0,1.0,-0.796198,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,3.0,-0.461587,-0.583386,1.0,0.0,13.0,4.0,5.0
16064,0.0,1.0,0.0,1.0,-0.518728,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,-0.461587,-0.583386,0.0,1.0,10.0,6.0,8.0
43640,1.0,1.0,1.0,1.0,-1.212402,0.0,0.0,1.0,1.0,1.0,...,1.0,0.0,4.0,1.985757,2.385074,1.0,1.0,13.0,4.0,5.0
38670,1.0,1.0,0.0,1.0,1.007355,1.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,-0.461587,-0.583386,0.0,1.0,8.0,5.0,8.0
30308,0.0,0.0,1.0,1.0,-0.379994,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,-0.461587,-0.583386,0.0,0.0,8.0,6.0,8.0


## Preparación de los Tensores 

Una vez que los datos han sido normalizados y codificados, es necesario prepararlos para el formato adecuado de PyTorch. Esto implica separar las características de la variable objetivo, convertirlas en tensores y organizarlas en un conjunto de datos. 

In [6]:
x_train = torch.tensor(df_train_norm.drop('Diabetes_binary', axis='columns').values).float()
x_test = torch.tensor(df_test_norm.drop('Diabetes_binary', axis='columns').values).float()

y_train = torch.tensor(df_train_norm['Diabetes_binary'].values).float()
y_test = torch.tensor(df_test_norm['Diabetes_binary'].values).float()

Una vez que los datos estan en formato tensor, se utiliza el modulo *TensorDataset* de PyTorch para agrupar los tensores en un único objeto

In [7]:
from torch.utils.data import DataLoader, TensorDataset

x_train = torch.tensor(df_train_norm.drop('Diabetes_binary', axis='columns').values).float()
x_test = torch.tensor(df_test_norm.drop('Diabetes_binary', axis='columns').values).float()


train_ds = TensorDataset(x_train, y_train)
batch_size = 8
torch.manual_seed(1)
train_dl = DataLoader(train_ds, batch_size, shuffle=True)

## Construcción del Modelo

El modelo es una red neuronal secuencias con capas densas (fully connected), usando funciones de activación ReLU y una capa final de salida. El modelo usa Mean Squared Error como función de pérdida y el optimizador SGD.

In [8]:
hidden_units = [8, 4]
input_size = x_train.shape[1]

all_layers = []
for hidden_unit in hidden_units:
    layer = nn.Linear(input_size, hidden_unit)
    all_layers.append(layer)
    all_layers.append(nn.ReLU())
    input_size = hidden_unit

all_layers.append(nn.Linear(hidden_units[-1], 1))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = nn.Sequential(*all_layers)
model = model.to(device)
model

Sequential(
  (0): Linear(in_features=21, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=4, bias=True)
  (3): ReLU()
  (4): Linear(in_features=4, out_features=1, bias=True)
)

In [9]:
loss_fn = nn.MSELoss()
#loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)


## Entrenamiento

El modelo se entrenó durante 100 epochs, con un registro cada 20 epochs. Cada iteración calcula la pérdida y ajusta los pesos.

In [10]:
torch.manual_seed(1)
num_epochs = 100
log_epochs = 20 
for epoch in range(num_epochs+1):
    loss_hist_train = 0
    for x_batch, y_batch in train_dl:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        pred = model(x_batch)[:, 0]
        loss = loss_fn(pred, y_batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        loss_hist_train += loss.item()
    if epoch % log_epochs==0:
        print(f'Epoch {epoch}  Loss {loss_hist_train/len(train_dl):.4f}')

Epoch 0  Loss 0.2183
Epoch 20  Loss 0.1720
Epoch 40  Loss 0.1708
Epoch 60  Loss 0.1698
Epoch 80  Loss 0.1690
Epoch 100  Loss 0.1683


## Evaluación del Modelo

Finalmente, se evalúa el modelo en el conjunto de prueba, obteniendo los resultados del Error Cuadrático Medio (Test MSE) que indica el promedio de error cuadrático en las predicciones; y el Error Absoluto Medio (Test MAE) que proporciona una media mas directa del error promedio.

In [11]:
with torch.no_grad():
    x_test = x_test.to(device)
    y_test = y_test.to(device)
    pred = model(x_test.float())[:, 0]
    loss = loss_fn(pred, y_test)
    print(f'Test MSE: {loss.item():.4f}')
    print(f'Test MAE: {nn.L1Loss()(pred, y_test).item():.4f}')

Test MSE: 0.1697
Test MAE: 0.3413


## Conclusiones

Realizamos pruebas con la función de pérdida nn.BCEWithLogitsLoss() y los resultados son cosniderablemnete peores a los obtenidos con la actual función de pérdida MSELoss.
Con el primer método mencionado obteníamos los siguientes valores en la evaluación del modelo:

    MSE 0.5113
    MAE 1.1527

Por ello decidimos quedarnos con la función de pérdida basada en el error cuadrático medio dados los resultados obtenidos.

Con esta nueva función de pérdida, el modelo fue evaluado en el conjunto de prueba obteniendo los resultados:

- MSE (Error cuadrático Medio): 0.1690
- MAE (Error Absoluto Medio): 0.3396

El valor MSE bajo indica que el modelo tiene un desempeño razonablemente bueno al predecir la presencia o ausencia de diabetes. El valor obtenido nos indica que las predicciones son bastantes cercanas a la realidad, aunque existe margen de mejora, posiblemente ajustando los hiperparametros.

En el caso del MAE, el valor obtenido indica que las predicciones tienen un error absoluto de aproximadamente 0.34, lo cual es aceptable considerando la variable objetiva binaria. 